If you're using IDE like VS Code, please install the **Jupyter** extension first.

After you run the **docker-compose up -d** command in your terminal (or copy and paste)


docker-compose up -d

Once the command is executed, type the following command in the terminal to get the token:

### Alternative
Run (copy and paste) the following command in a terminal to generate a token, then paste it into the link below.

http://127.0.0.1:8888/lab?token=XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX

docker exec jupyter_analytics jupyter server list

### Recommended
Run the above command, copy the output link NOT INCLUDED **:: //home/jovyan**. 

Go to your notebook and click **Select Kernel** in the top-right corner.

Go to **Select Another Kernel...** > **Existing Jupyter Server...** and paste **output link** and replace the number and letter combination after the http:// with **localhost** in the input box.

e.g. http://5ab7977b7ef1:8888/?token=28737xxxxxxxxxxxxxxxxxxxxxxxxxxxxcf55d5 

➡️ http://localhost:8888/?token=28737xxxxxxxxxxxxxxxxxxxxxxxxxxxxcf55d5

Hit Enter, then Enter again... and you're in!

🚨 AND DO **NOT** FORGET to run the **First cell** before you do anything else!

##### Before you start processing the data, please download pymongo package.

In [1]:
%pip install pymongo

Note: you may need to restart the kernel to use updated packages.


#### Predefine Functions: Make sure HDFS and MongoDB are cleaned

In [2]:
def manage_hdfs_path(path="/ecommerce", execute_delete=True):
    print(f"🚀 Starting HDFS Management for: {path}\n")
    try:
        # Initialize FileSystem ONLY ONCE
        fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
        target_path = spark._jvm.org.apache.hadoop.fs.Path(path)
        
        # --- Internal Helper: Recursive Tree Printer ---
        def print_tree(current_path, prefix=""):
            try:
                statuses = fs.listStatus(current_path)
                sorted_statuses = sorted(statuses, key=lambda s: (not s.isDirectory(), s.getPath().getName()))
                
                for i, status in enumerate(sorted_statuses):
                    is_last = (i == len(sorted_statuses) - 1)
                    marker = "└── " if is_last else "├── "
                    item_name = status.getPath().getName()
                    
                    if status.isDirectory():
                        print(f"{prefix}{marker}[DIR] {item_name}/")
                        new_prefix = prefix + ("    " if is_last else "│   ")
                        print_tree(status.getPath(), new_prefix)
                    else:
                        item_size = status.getLen()
                        print(f"{prefix}{marker}[FILE] {item_name} ({item_size} bytes)")
            except Exception as e:
                print(f"{prefix}└── ❌ Error accessing path: {e}")

        # --- Step 1: Inspect the Path ---
        print(f"🔍 Inspecting tree structure...")
        if fs.exists(target_path):
            print(f"[DIR] {path}/")
            print_tree(target_path)
            print("✅ Tree generation complete.\n")
            
            # --- Step 2: Delete the Path ---
            if execute_delete:
                print(f"🧹 Wiping HDFS path {path}...")
                fs.delete(target_path, True)
                print("✅ HDFS Cleaned successfully.")
            else:
                print(f"⏸️ execute_delete is False. Skipping deletion.")
                
        else:
            print(f"⚠️ Path '{path}' does not exist. Nothing to inspect or clean.")

    except Exception as e:
        print(f"❌ Error during HDFS operations: {e}")

In [5]:
def clean_mongo_db(db_name="amazon_reviews"):
    print(f"🔍 Checking MongoDB for database: '{db_name}'...")
    client = MongoClient("mongodb://mongodb:27017/")
    
    try:
        # 1. Check if the database exists
        existing_dbs = client.list_database_names()
        
        if db_name not in existing_dbs:
            print(f"⚠️ Database '{db_name}' does not exist! Nothing to delete.\n")
            return
            
        # 2. If it exists, show its collections first (good for logging)
        db = client[db_name]
        collections = db.list_collection_names()
        print(f"📁 Found database '{db_name}' with collections: {collections}")
        
        # 3. Execute the deletion
        print(f"🗑️ Dropping database '{db_name}'...")
        client.drop_database(db_name)
        print(f"✅ Database '{db_name}' successfully deleted.\n")
        
    except Exception as e:
        print(f"❌ Error occurred: {e}\n")
    finally:
        # Always close the connection
        client.close()

# PHASE 1 & 2 Data Preprocessing

In [1]:
# MUST RUN, OR RUN AFTER YOU ARE REFUSED
# ==========================================
# Spark Session Init
# ==========================================
import os
from pyspark.sql import SparkSession

# STEP 1: Set working directory & HDFS identity
os.environ['HADOOP_USER_NAME'] = 'root'
work_dir = '/home/jovyan/work'
if os.path.exists(work_dir):
    os.chdir(work_dir)
print(f"📁 Working Directory: {os.getcwd()}")
print(f"📂 Files found: {os.listdir('.')}")

# STEP 2: Stop any existing Spark session to free memory
existing = SparkSession.getActiveSession()
if existing:
    print(f"🧹 Stopping existing Spark session: {existing.sparkContext.appName}")
    existing.stop()
    print("✅ Old session stopped — JVM memory released.")
else:
    print("ℹ️ No existing Spark session found.")
    
# STEP 3: Start fresh Spark session
# Initialize Spark session with MongoDB connector JARs
MONGO_JARS = ",".join([
    "/home/jovyan/.ivy2/cache/org.mongodb.spark/mongo-spark-connector_2.12/jars/mongo-spark-connector_2.12-10.3.0.jar",
    "/home/jovyan/.ivy2/cache/org.mongodb/mongodb-driver-sync/jars/mongodb-driver-sync-4.8.2.jar",
    "/home/jovyan/.ivy2/cache/org.mongodb/bson/jars/bson-4.8.2.jar",
    "/home/jovyan/.ivy2/cache/org.mongodb/mongodb-driver-core/jars/mongodb-driver-core-4.8.2.jar",  # 4.0.5 → 4.8.2,
])

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ecommerce_pipeline") \
    .master("local[4]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "false") \
    .getOrCreate()

spark.sparkContext.setCheckpointDir("hdfs://namenode:9000/tmp/spark_checkpoint")

sc = spark.sparkContext
print(f"✅ Spark Ready!")
print(f"   App Name : {sc.appName}")
print(f"   Version  : {sc.version}")
print(f"   Master   : {sc.master}")
print(f"   Spark UI : http://localhost:4040")

# Check if there is a mongo jar in the container
import glob
all_jars = glob.glob("/home/jovyan/.ivy2/**/*.jar", recursive=True) + \
           glob.glob("/usr/local/spark/jars/**/*.jar", recursive=True)
mongo_jars = [j for j in all_jars if "mongo" in j.lower()]
print("🔍 All mongo jars found:")
for j in mongo_jars:
    print(f"   {j}")

📁 Working Directory: /home/jovyan/work
📂 Files found: ['.git', '.gitignore', '.ipynb_checkpoints', 'configs', 'data', 'data_lake', 'docker-compose.yml', 'files', 'hadoop.env', 'metadata_full_all_columns.xlsx', 'meta_cluster.xlsx', 'meta_cluster_w2v.xlsx', 'meta_review.xlsx', 'pngs', 'spark-warehouse', 'venv', '[Phase1&2]_PreProcess.ipynb', '[Phase3&4]_Storage&Analytics.ipynb']
ℹ️ No existing Spark session found.
✅ Spark Ready!
   App Name : ecommerce_pipeline
   Version  : 3.5.0
   Master   : local[4]
   Spark UI : http://localhost:4040
🔍 All mongo jars found:


In [5]:
# clear HDFS
manage_hdfs_path()

from pymongo import MongoClient
# clear MongoDB
clean_mongo_db()

🚀 Starting HDFS Management for: /ecommerce

🔍 Inspecting tree structure...
⚠️ Path '/ecommerce' does not exist. Nothing to inspect or clean.
🔍 Checking MongoDB for database: 'amazon_reviews'...
📁 Found database 'amazon_reviews' with collections: ['review_cellp', 'review_elec', 'meta_elec', 'meta_cellp']
🗑️ Dropping database 'amazon_reviews'...
✅ Database 'amazon_reviews' successfully deleted.



In [6]:
# Utlize MongoDB to store metadata and reviews
# ==========================================================
import os, shutil, sys, json, time
from pymongo import MongoClient

config_file = "configs/datasets.txt"
output_folder = "data_lake"

time_rd_start = time.time()

# --- Path diagnostics ---
print(f"📁 CWD: {os.getcwd()}")
print(f"🔍 Looking for config: {os.path.abspath(config_file)}")
print(f"📂 Files in CWD: {os.listdir('.')}")


# Ensure local data lake folder exists
os.makedirs(output_folder, exist_ok=True)



if not os.path.exists(config_file):
    print(f"\n❌ datasets.txt not found at: {os.path.abspath(config_file)}")
    print("   👉 Make sure datasets.txt is in your mounted project root.")
else:
    with open(config_file, 'r') as f:
        lines = [line.strip() for line in f if line.strip()]

    print(f"\n📋 Found {len(lines)} dataset(s) to process: {lines}")
   
    for i, line in enumerate(lines):
        dataset_name, table_name = [item.strip() for item in line.split(',')]

        print(f"\n{'='*60}")
        print(f"🔄 [{i+1}/{len(lines)}] Dataset: [{dataset_name}] → table: [{table_name}]")
        print(f"{'='*60}")

        meta_local_in  = os.path.abspath(f"data/meta/meta_{dataset_name}.jsonl")
        rev_local_in   = os.path.abspath(f"data/review/{dataset_name}.jsonl")
    
        tasks = [
            {
                "label": "META",
                "file":  meta_local_in,
                "col":   f"meta_{table_name}",
            },
            {
                "label": "REV",
                "file":  rev_local_in,
                "col":   f"review_{table_name}",
            },
        ]
        # ---------------------------------------------------------
        # TASK A & B: Metadata + Reviews → MongoDB (FULL load)
        # ---------------------------------------------------------
        for task in tasks:
            label  = task["label"]
            infile = task["file"]
            col_name = task["col"]

            if not os.path.exists(infile):
                print(f"⚠️  [{label}] File not found, skipping: {infile}")
                continue

            # clear old collection
            try:
                client = MongoClient("mongodb://mongodb:27017/")
                db = client["amazon_reviews"]
                if col_name in db.list_collection_names():
                    count_before = db[col_name].count_documents({})
                    db[col_name].drop()
                    print(f"🗑️  [{label}] '{col_name}' dropped — {count_before:,} docs removed.")
                else:
                    print(f"ℹ️  [{label}] '{col_name}' not found — fresh write.")
                client.close()
            except Exception as e:
                print(f"❌ [{label}] Pre-clear failed: {e}")
                continue

            # write to MongoDB
            try:
                BATCH_SIZE = 50000

                client = MongoClient(
                    "mongodb://mongodb:27017/",
                    maxPoolSize=10,
                    socketTimeoutMS=60000,
                    connectTimeoutMS=10000
                )
                collection = client["amazon_reviews"][col_name]

                file_size = os.path.getsize(infile)
                print(f"\n📦 [{label}] Reading: {infile}")
                print(f"   File size : {file_size / (1024**3):.2f} GB")

                total_written = 0
                skipped = 0
                batch = []
                bytes_read = 0

                with open(infile, 'r', encoding='utf-8') as f:
                    for line in f:
                        bytes_read += len(line.encode('utf-8'))
                        line = line.strip()
                        if not line:
                            continue
                        try:
                            batch.append(json.loads(line))
                            if len(batch) >= BATCH_SIZE:
                                collection.insert_many(batch, ordered=False)
                                total_written += len(batch)
                                batch = []

                                pct = min(int(bytes_read / file_size * 100), 99)
                                bar = "█" * (pct // 5) + "░" * (20 - pct // 5)
                                sys.stdout.write(f"\r   ⏳ [{bar}] {pct}% (~{total_written:,} rows)")
                                sys.stdout.flush()
                        except json.JSONDecodeError:
                            skipped += 1

                # last batch
                if batch:
                    collection.insert_many(batch, ordered=False)
                    total_written += len(batch)

                client.close()
                print(f"\n✅ [{label}] Done!")
                print(f"   Written  : {total_written:,}")
                print(f"   Skipped  : {skipped:,} (malformed JSON)")

            except Exception as e:
                import traceback
                traceback.print_exc()
            finally:
                spark.catalog.clearCache()
                print(f"🧹 [{label}] Cache cleared.")

print(f"\n✨ ALL DUAL-TRACK INGESTION TASKS COMPLETED! Total time: {time.time()-time_rd_start:.1f}s")

📁 CWD: /home/jovyan/work
🔍 Looking for config: /home/jovyan/work/configs/datasets.txt
📂 Files in CWD: ['.git', '.gitignore', '.ipynb_checkpoints', 'configs', 'data', 'data_lake', 'docker-compose.yml', 'files', 'hadoop.env', 'metadata_full_all_columns.xlsx', 'meta_cluster.xlsx', 'meta_cluster_w2v.xlsx', 'meta_review.xlsx', 'pngs', 'spark-warehouse', 'venv', '[Phase1&2]_PreProcess.ipynb', '[Phase3&4]_Storage&Analytics.ipynb']

📋 Found 2 dataset(s) to process: ['Electronics, elec', 'Cell_Phones_and_Accessories, cellp']

🔄 [1/2] Dataset: [Electronics] → table: [elec]
ℹ️  [META] 'meta_elec' not found — fresh write.

📦 [META] Reading: /home/jovyan/work/data/meta/meta_Electronics.jsonl
   File size : 4.89 GB
   ⏳ [███████████████████░] 99% (~1,600,000 rows)
✅ [META] Done!
   Written  : 1,610,012
   Skipped  : 0 (malformed JSON)
🧹 [META] Cache cleared.
ℹ️  [REV] 'review_elec' not found — fresh write.

📦 [REV] Reading: /home/jovyan/work/data/review/Electronics.jsonl
   File size : 21.06 GB
   ⏳

In [8]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


### ⚠️ Troubleshooting Note
If you encounter **any errors** (such as `EOFError`, Py4J disconnects, or OOM failures) during preprocessing:
1. **Restart the Notebook Kernel** to clear the memory.
2. **Re-run only the first cell** to re-initialize the Spark session.
3. **Re-run the specific cell** where the error occurred and below using **Execute Cell and  Below**.

*(DO NOT re-run the completed cells.)*

In [2]:
# Brand Whitelist Filter → Metadata Clean → Review Alignment (Chunked)
# ==========================================

import os, sys
import time
import json
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pymongo import MongoClient
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType, StructType, StructField, StringType, IntegerType, BooleanType, LongType

HDFS_META_OUT   = "hdfs://namenode:9000/ecommerce/data_lake/meta"
HDFS_REVIEW_OUT = "hdfs://namenode:9000/ecommerce/data_lake/review"

# Ready to send file to partners
# CREATEFILE = True

# Testing code completeness
CREATEFILE = False

BRAND_WHITELIST = [
    # Smartphones / Tablets
    "apple", "samsung", "google", "sony", "oneplus", "xiaomi", "oppo", "huawei",
    "motorola", "nokia", "realme", "vivo", "zte", "blackberry", "honor",
    # Laptops / PCs
    "dell", "hp", "lenovo", "asus", "acer", "microsoft", "razer", "msi",
    "toshiba", "gigabyte", "lg", "alienware", "system76",
    # Audio
    "bose", "jbl", "sennheiser", "jabra", "anker", "soundcore",
    "beats", "shure", "audio-technica", "skullcandy",
    "plantronics", "beyerdynamic", "marshall",
    # Gaming Peripherals
    "logitech", "corsair", "steelseries", "hyperx", "roccat",
    "zowie", "glorious", "ducky",
    # Cameras
    "canon", "nikon", "fujifilm", "gopro",
    "panasonic", "olympus", "leica", "dji", "insta360", "sigma", "pentax",
    # Displays / TVs
    "tcl", "hisense", "vizio", "philips",
    "viewsonic", "benq", "aoc",
]
BRAND_SET_LOWER = set(b.lower() for b in BRAND_WHITELIST)

# in previous work, already know which columns are important for analysis
META_COLS = ["parent_asin", "title", "price", "store", "main_category",
             "average_rating", "rating_number", "details"]

REV_SCHEMA = StructType([
    StructField("parent_asin",       StringType(),  True),
    StructField("asin",              StringType(),  True),
    StructField("rating",            FloatType(),   True),
    StructField("title",             StringType(),  True),
    StructField("text",              StringType(),  True),
    StructField("timestamp",         LongType(),    True),
    StructField("helpful_vote",      IntegerType(), True),
    StructField("verified_purchase", BooleanType(), True),
    StructField("user_id",           StringType(),  True),
])
REV_COLS = [f.name for f in REV_SCHEMA.fields]

# ─────────────────────────────────────────
# Dataset loop
# ─────────────────────────────────────────
time_ft_start = time.time()
with open("configs/datasets.txt", 'r') as f:
    lines = [line.strip() for line in f if line.strip()]

for idx, line in enumerate(lines):
    dataset_name, table_name = [x.strip() for x in line.split(',')]
    t0 = time.time()
    print(f"\n{'='*60}")
    print(f"🔄 [{idx+1}/{len(lines)}] Processing: [{dataset_name}] → [{table_name}]")
    print(f"{'='*60}")

    client = MongoClient(
        "mongodb://mongodb:27017/",
        directConnection=True,
        serverSelectionTimeoutMS=30000
    )
    db = client["amazon_reviews"]

    # ── STEP 1: Load metadata from MongoDB ────────────────────
    meta_col_name = f"meta_{table_name}"
    print(f"\n📦 [META] Reading MongoDB collection: {meta_col_name}")

    total_docs = db[meta_col_name].count_documents({})
    print(f"   Total docs : {total_docs:,}")

    CHUNK_SIZE = 100000
    batch_data = []
    total_fetched = 0
    df_meta = None
    t_meta = time.time()

    meta_cursor = db[meta_col_name].find(
        {}, {"_id": 0, **{col: 1 for col in META_COLS}}
    ).batch_size(CHUNK_SIZE)

    for doc in meta_cursor:
        if "details" in doc and isinstance(doc["details"], dict):
            doc["details"] = json.dumps(doc["details"])
        batch_data.append(doc)
        total_fetched += 1

        if len(batch_data) >= CHUNK_SIZE:
            chunk_df = spark.createDataFrame(pd.DataFrame(batch_data))
            df_meta = chunk_df if df_meta is None else df_meta.union(chunk_df)
            batch_data = []

            pct = min(int(total_fetched / total_docs * 100), 99)
            bar = "█" * (pct // 5) + "░" * (20 - pct // 5)
            sys.stdout.write(f"\r   ⏳ [{bar}] {pct}% ({total_fetched:,} / {total_docs:,})")
            sys.stdout.flush()

    if batch_data:
        chunk_df = spark.createDataFrame(pd.DataFrame(batch_data))
        df_meta = chunk_df if df_meta is None else df_meta.union(chunk_df)

    print(f"\n   ✅ Done! Total fetched: {total_fetched:,}")
    
    count_before = df_meta.count()
    df_meta = df_meta.checkpoint()
    print(f"✅ [META] Checkpoint done.")

    # ── STEP 2: Brand whitelist filter ────────────────────────
    df_meta_filtered = df_meta.filter(
        F.lower(F.col("store")).isin(list(BRAND_SET_LOWER))
    )
    
    count_after  = df_meta_filtered.count()
    print(f"   Rows before : {count_before:,}")
    print(f"   Rows after  : {count_after:,}  ({count_after/count_before*100:.1f}% kept)")
    print(f"✅ [META] Fetched {total_fetched:,} metadata in {time.time()-t_meta:.1f}s")

    # ── STEP 3: Select columns + cast ─────────────────────────
    df_meta_clean = df_meta_filtered.select(
        "parent_asin",
        "title",
        F.col("price").cast(FloatType()).alias("price"),
        "store",
        "main_category",
        F.col("average_rating").cast(FloatType()).alias("average_rating"),
        "rating_number",
        "details",
    )

    # ── STEP 4: Collect valid parent_asin ─────────────────────
    valid_asins = [
        row["parent_asin"]
        for row in df_meta_clean.select("parent_asin").distinct().collect()
    ]
    print(f"   Valid parent_asin: {len(valid_asins):,}")

    # ── STEP 5: Save metadata → HDFS ──────────────────────────
    meta_hdfs_path = f"{HDFS_META_OUT}/{table_name}"
    print(f"\n💾 [META] Saving → HDFS: {meta_hdfs_path}")
    df_meta_clean.repartition(max(1, count_after // 20000)) \
        .write.mode("overwrite") \
        .parquet(meta_hdfs_path)
    print(f"✅ [META] Saved.")

    if CREATEFILE:
        local_meta_path = f"file://{os.path.abspath(f'data_lake/meta_{table_name}.parquet')}"
        print(f"💾 [META] Local backup → {local_meta_path}")
        df_meta_clean.write.mode("overwrite").parquet(local_meta_path)
        print(f"✅ [META] Local backup saved.")

    # ── STEP 6: Chunked review extraction → HDFS ──────────────
    rev_col_name = f"review_{table_name}"
    print(f"\n📦 [REV] Querying MongoDB collection filtered by parent_asin of metadata: {rev_col_name}")
    t_rev = time.time()

    total_rev_docs = db[rev_col_name].count_documents({"parent_asin": {"$in": valid_asins}})
    print(f"   Total docs : {total_rev_docs:,}")

    # make sure only fetch reviews for valid parent_asin in meta_clean set
    rev_cursor = db[rev_col_name].find(
        {"parent_asin": {"$in": valid_asins}},
        {"_id": 0, **{col: 1 for col in REV_COLS}}
    ).batch_size(50000)

    rev_hdfs_path = f"{HDFS_REVIEW_OUT}/{table_name}"

    CHUNK_SIZE = 50000
    batch_data = []
    total_fetched = 0
    df_rev = None

    for doc in rev_cursor:
        batch_data.append({k: doc.get(k) for k in REV_COLS})
        total_fetched += 1

        if len(batch_data) >= CHUNK_SIZE:
            chunk_df = spark.createDataFrame(pd.DataFrame(batch_data))
            df_rev = chunk_df if df_rev is None else df_rev.union(chunk_df)
            batch_data = []

            pct = min(int(total_fetched / total_rev_docs * 100), 99)
            bar = "█" * (pct // 5) + "░" * (20 - pct // 5)
            sys.stdout.write(f"\r   ⏳ [{bar}] {pct}% ({total_fetched:,} / {total_rev_docs:,})")
            sys.stdout.flush()

    if batch_data:
        chunk_df = spark.createDataFrame(pd.DataFrame(batch_data))
        df_rev = chunk_df if df_rev is None else df_rev.union(chunk_df)

    print(f"\n   ✅ Done! Total fetched: {total_fetched:,}")

    df_rev = df_rev.checkpoint()
    print(f"✅ [REV] Checkpoint done.")
    print(f"✅ [REV] Fetched {total_fetched:,} reviews in {time.time()-t_rev:.1f}s")

    # write to HDFS
    print(f"💾 [REV] Saving → HDFS: {rev_hdfs_path}")
    df_rev.repartition(max(1, total_fetched // 20000)) \
        .write.mode("overwrite") \
        .parquet(rev_hdfs_path)
    print(f"✅ [REV] Saved.")

    if CREATEFILE:
        local_rev_path = f"file://{os.path.abspath(f'data_lake/review_{table_name}.parquet')}"
        print(f"💾 [REV] Local backup → {local_rev_path}")
        df_rev.write.mode("overwrite").parquet(local_rev_path)
        print(f"✅ [REV] Local backup saved.")

    # Cleanup
    df_meta.unpersist()
    df_meta_clean.unpersist()
    df_rev.unpersist()
    spark.catalog.clearCache()
    print(f"\n⏱️  [{dataset_name}] Total time: {time.time()-t0:.1f}s")

time_ft_end = time.time()
print(f"\n✨ ALL DATASETS CLEANED AND SAVED! Total time: {time_ft_end-time_ft_start:.1f}s")


🔄 [1/2] Processing: [Electronics] → [elec]

📦 [META] Reading MongoDB collection: meta_elec
   Total docs : 1,610,012
   ⏳ [███████████████████░] 99% (1,600,000 / 1,610,012)
   ✅ Done! Total fetched: 1,610,012
✅ [META] Checkpoint done.
   Rows before : 1,610,012
   Rows after  : 128,428  (8.0% kept)
✅ [META] Fetched 1,610,012 metadata in 230.0s
   Valid parent_asin: 128,428

💾 [META] Saving → HDFS: hdfs://namenode:9000/ecommerce/data_lake/meta/elec
✅ [META] Saved.

📦 [REV] Querying MongoDB collection filtered by parent_asin of metadata: review_elec
   Total docs : 5,984,931
   ⏳ [███████████████████░] 99% (5,950,000 / 5,984,931)
   ✅ Done! Total fetched: 5,984,931
✅ [REV] Checkpoint done.
✅ [REV] Fetched 5,984,931 reviews in 1127.5s
💾 [REV] Saving → HDFS: hdfs://namenode:9000/ecommerce/data_lake/review/elec
✅ [REV] Saved.

⏱️  [Electronics] Total time: 1574.5s

🔄 [2/2] Processing: [Cell_Phones_and_Accessories] → [cellp]

📦 [META] Reading MongoDB collection: meta_cellp
   Total docs : 1

In [3]:
# For procdure completeness
# if set not to backup local files
# then not going to read local datafile later (in Phase 3&4)
if os.path.isfile('configs/local'):
    # delete signal file in advance
    os.remove('configs/local')

if CREATEFILE == False:
    with open(f'configs/local', 'w') as f:
        f.write('')

In [6]:
clean_mongo_db()


🔍 Checking MongoDB for database: 'amazon_reviews'...
📁 Found database 'amazon_reviews' with collections: ['review_cellp', 'meta_elec', 'meta_cellp', 'review_elec']
🗑️ Dropping database 'amazon_reviews'...
✅ Database 'amazon_reviews' successfully deleted.

